# 实验23-v3：单层波形Transformer与实时梯度监控

本笔记本是正式训练入口，调用 `experiment_v3.py`，不会复制或改变训练逻辑。运行下一单元后将：

- 按种子20260826、五个测试fold训练或续跑共5个模型折（最多50轮，patience=5）；
- 在Windows桌面弹出独立的Matplotlib/Tkinter实时梯度监控窗口；关闭该窗口不会停止训练；
- 启动后立即返回；训练日志写入输出目录，实时曲线在独立窗口显示。

请在具备桌面图形界面的本机Jupyter内核中运行。不要在已有同一实验训练进程运行时再次执行本单元。

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
SCRIPT = ROOT / "23_BCSCStructureClickClassification" / "experiment_v3.py"
OUTPUT = ROOT / "23_BCSCStructureClickClassification" / "v3_waveform_1layer_gradient_seed20260826_epoch50"

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

# `all` 会先验证、训练/续跑5折，再仅基于实际结果生成报告和图形。
# 保持此项为False即可显示独立监控窗口。
NO_LIVE_MONITOR = False
command = [sys.executable, str(SCRIPT), "all"]
if NO_LIVE_MONITOR:
    command.append("--no-live-monitor")

print("启动命令：", " ".join(command))
print("实时事件文件：", OUTPUT / "live" / "events.jsonl")
print("监控窗口会作为独立桌面窗口出现；关闭窗口不会停止此单元的训练。")

# 不在Notebook中持续读取stdout：这会在内核被停止时显示KeyboardInterrupt。
# 子进程将日志写入文件，单元会立即返回；实时曲线由独立监控窗口显示。
LOG_PATH = OUTPUT / "live" / "training_console.log"
PID_PATH = OUTPUT / "live" / "training.pid"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

with LOG_PATH.open("a", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command, cwd=str(ROOT), stdout=log_file, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace"
    )

PID_PATH.write_text(str(process.pid), encoding="utf-8")
print(f"训练已启动，进程ID：{process.pid}")
print("本单元已返回；请查看独立监控窗口。")
print("训练日志：", LOG_PATH)
print("训练结束后，可重新运行本单元续跑，或执行 report 生成报告。")


## 监控窗口说明

窗口标题为“实验23-v3 实时梯度监控（关闭不停止训练）”。它每2秒从 `live/events.jsonl` 读取完整事件行，显示训练loss、验证低SNR AUROC、全局L2/RMS、8类模块RMS和健康状态。

若本机没有图形桌面，训练仍继续，且可把 `NO_LIVE_MONITOR` 改为 `True`。